<a href="https://colab.research.google.com/github/nawrotlab/Nawrot_CNS_Course_solutions/blob/main/Trial_by_trial_variability_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

All data was kindly provided by Alexa Riehle, Institut de Neurosciences de la Timone (INT),
Centre National de la Recherche Scientifique (CNRS) - Aix-Marseille Université (AMU),
UMR7289, 13005 Marseille, France. Use of this data outside of this teaching course is strictly
prohibited.

# **Trial-by-trial variability and firing irregularity in single unit spike trains from monkey motor cortex.** - Solutions

*Martin Nawrot*

This course module introduces the student to second order statistical analyses of
single neuron spike trains. The exercises cover the time-resolved measurement of
trial-by-trial variability using the Fano factor (FF) and the estimation of interval
variability using the local coefficient of variation (CV2). The results on interval and
count variability are interpreted in the context of predictions from point process
theory. This teaching module addresses graduate students with a solid background
in programming. The module is designed for one course day including an
introductory lecture, practical course work, and final presentation of results.

### Data sets

In the present exercises, you will analyze single unit recordings from the primary motor cortex (M1) of one monkey (monkey 1 in Rickert et al., 2009), which performed a delayed center-out task. The experiments were carried out in the lab of Alexa Riehle. The task and the data are described in detail in Rickert et al. (2009). For an introduction to directional tuning in the motor cortex, see Riehle & Vaadia (2005).

All motor-cortex datasets used in this notebook are stored inside `data/motor_cortex/` in this repository. In Colab, run the setup cell below once; it clones the repository and defines `DATA_DIR`, `SPARSE_DATA_DIR`, `SELECTED_C1_DIR`, and `SELECTED_C3_DIR`.

The files are MATLAB `.mat` files. In Python, a robust loading pattern is:

```python
from scipy.io import loadmat
mat = loadmat(path, squeeze_me=True, struct_as_record=False)
sparse_format = mat["SparseFormat"]
spike_matrices = sparse_format.Data
time_resolution_ms = sparse_format.TimeResolutionMS
cut_interval_ms = sparse_format.CutIntervalMS
```

`spike_matrices` is a length-6 array. Each entry corresponds to one movement direction and is stored as a sparse matrix with shape `(time_bins, n_trials)`. Convert a sparse matrix to a dense NumPy array with `.toarray()` when needed.


In [ ]:

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/schmitfe/Nawrot_CNS_Course.git"
REPO_NAME = "Nawrot_CNS_Course"


def prepare_repo() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data").is_dir():
        return cwd

    clone_parent = Path("/content") if "google.colab" in sys.modules else cwd
    repo_root = clone_parent / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    os.chdir(repo_root)
    return repo_root.resolve()


REPO_ROOT = prepare_repo()
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as spio
import scipy.signal

DATA_DIR = REPO_ROOT / "data" / "motor_cortex"
SPARSE_DATA_DIR = DATA_DIR / "data_sparse_FromLinux"
SELECTED_C1_DIR = DATA_DIR / "SelectedDataC1"
SELECTED_C3_DIR = DATA_DIR / "SelectedDataC3"


In [ ]:
SELECTED_C1_TS_FILES = sorted(SELECTED_C1_DIR.glob("*-C1-TS.mat"))
SELECTED_C3_TS_FILES = sorted(SELECTED_C3_DIR.glob("*-C3-TS.mat"))

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"Selected C1 TS files: {len(SELECTED_C1_TS_FILES)}")
print(f"Selected C3 TS files: {len(SELECTED_C3_TS_FILES)}")
print(f"Example C1 TS file: {SELECTED_C1_TS_FILES[0].name}")


# **Exercises**

The present teaching module is designed for 2-3 hours of practical work and divides
into two parts. The goal of this module is to finish each part properly, no matter
whether all parts will be completed. Students will present and discuss their results on
individual parts at the end of class.

### **Part I – Time resolved Fano factor estimation**

Neural activity is inherently variable across identical experimental repetitions and this
variability is typically high with respect to the trial-averaged mean response (e.g. Arieli et al.,
1996). In spike trains that were measured across repeated experimental trials we can
measure the trial-by-trial variability of the single neuron output (e.g. Shadlen and Newsome,
1998). The most established measure is the Fano factor defined as the variance of the spike
count across trials divided by the mean spike count (for a practical introduction and
interpretation of results see Nawrot 2010). This measure has interpretational advantages in
the context of stochastic point process theory. Trial-by-trial spike count is particularly high in
the neocortex and there it is typically highest in the motor areas. However, trial-by-trial
variability is modulated in a task-related manner which was first shown in motor cortex
(Churchland M et al., 2006; Nawrot et al., 2008; Rickert et al., 2009) but is a signature in all
cortical areas (Churchland M et al., 2010; Churchland A et al., 2011). For the present data
set this has been demonstrated in Nawrot et al. (2003) and in the supplements of Rickert et
al. (2009).

In this part of the exercises we will use the kernel method to estimate the time-resolved Fano
factor averaged over a population of neurons.

Tasks
1. The results shall be presented in *Figure 2* with 2 x 1 panels (2 row panels). You may use `subplots` from `matplotlib.pyplot` to generate several axes within one figure.
2. Choose one single data file with spike trains aligned to either TS or MO and load it into memory. A good starting point is one file from `SELECTED_C1_DIR` or `SELECTED_C3_DIR`. Select one out of the six movement directions. What is the time resolution of the binary spike representation?
3. *Computation of the Fano factor*. For a single-unit data file, compute the mean spike count across trials and the variance of the spike count across trials. To this end, count the number of spikes in a moving window. This is conveniently accomplished by applying kernel convolution with a rectangular kernel *k* to all single trials. If the entries in the boxcar kernel are all equal to 1, this results in counting the number of spikes per window (hint: you can use `np.ones` for constructing *k*). The spike count equals the number of spikes within the window of the kernel.
   
   You may convolve the full spike matrix (referred to as `spike_data`) at once with the kernel array *k* by using the `lfilter` function from `scipy.signal`: `R = scipy.signal.lfilter(k, 1.0, spike_data, axis=1)`. Check what happens at the left and right edges of the signal. If needed, shift the result so that the count is associated with the center of the window.


### Visual intuition: centered boxcar convolution

The animation below shows a single spike train together with a **centered** boxcar kernel. At each time bin, the kernel is centered on that bin, the spikes inside the window are counted, and the resulting value is assigned to the **center position**. This is exactly the moving-window count used for the Fano-factor analysis.

![Centered boxcar convolution](assets/kernel_convolution_centered.gif)

If you use a boxcar kernel filled with ones, the convolution returns spike **counts** inside the window. If you use `scipy.signal.lfilter`, remember that the raw output is aligned causally; shift it afterward so that the count is associated with the center of the window.


4. As a result, you obtain a time-resolved estimate of the spike count for each single trial. In a next step compute the variance, the mean across trials, and the Fano factor. Plot the resulting function $FF(t)$ in the top panel of *Figure 2*. What is a good window width *w* for the boxcar kernel? Try out different single-unit data files. What happens if the trial-averaged spike count drops to zero?
5. In a next step we want to estimate the average Fano factor across different directions and many single units (as in Rickert et al., 2009 or Churchland et al., 2010).  
   To this end, modify your script to (i) loop across all six directions for each data file, and (ii) loop across a pre-defined list of data files. Choose one experimental condition C1, C2, or C3 and data aligned either to TS or MO.  
   Compute, for each single unit, the Fano factor for each movement direction and average $FF$ across directions. Then average across all single units and plot the grand average of the FF in panel 2 of *Figure 2*.
6. Are your results consistent with the results in Churchland et al., 2010 (https://www.nature.com/articles/nn.2501) and Rickert et al., 2009 (https://www.jneurosci.org/content/29/19/6053)? Which similarities or differences do you observe?


Given the vector v containing the observed spike counts (one per spike train) in the time window [t0, t1], $FF$ is defined as:
$$ FF := \frac{var(v)}{mean(v)} $$

In [ ]:
def load_sparse_format(path):
    return spio.loadmat(path, squeeze_me=True, struct_as_record=False)["SparseFormat"]


def centered_counts(spike_data, window_bins):
    kernel = np.ones((window_bins, 1))
    return scipy.signal.convolve(spike_data, kernel, mode="same")


def fano_factor(counts):
    mean = counts.mean(axis=1)
    variance = counts.var(axis=1)
    return np.divide(variance, mean, out=np.full_like(mean, np.nan), where=mean > 0)


example = load_sparse_format(SELECTED_C1_TS_FILES[0])
spike_data = example.Data[0].toarray()
dt_ms = float(example.TimeResolutionMS)
time_ms = np.arange(spike_data.shape[0]) * dt_ms + example.CutIntervalMS[0]
window_bins = round(400 / dt_ms)
example_ff = fano_factor(centered_counts(spike_data, window_bins))
print("binary spike-train resolution (ms):", dt_ms)

unit_ff = []
for path in SELECTED_C1_TS_FILES:
    sparse = load_sparse_format(path)
    direction_ff = []
    for direction_data in sparse.Data:
        spike_data = direction_data.toarray()[:time_ms.size]
        counts = centered_counts(spike_data, window_bins)
        direction_ff.append(fano_factor(counts))
    unit_ff.append(np.nanmean(direction_ff, axis=0))
grand_ff = np.nanmean(unit_ff, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(time_ms, example_ff)
axes[0].set_title("one unit, one direction")
axes[1].plot(time_ms, grand_ff)
axes[1].set_title("mean across C1 units and directions")
for ax in axes:
    ax.set_ylabel("Fano factor")
axes[1].set_xlabel("time relative to TS (ms)")
plt.tight_layout()
plt.show()


### **Part II – Interval versus count variability of neural spike trains**

In this part we will directly compare the variability of inter-spike intervals (ISI) with the
trial-by-trial count variability of single units. This allows us to interpret the results in
the light of stochastic point process theory. We have already introduced the Fano
factor ($FF$) as a measure of trial-by-trial variability. In addition we here introduce the
coefficient of variation ($CV$) of the lengths of inter-spike intervals which is defined as
the ratio of standard deviation divided by the mean, i.e. a dimensionless number,
hence a coefficient. The $CV$ is a measure for the irregularity of the spike train and
sometimes spike train irregularity is used synonymously with inter-spike interval
variability. For the broad class of renewal processes, on expectation the two random
variables are approximately related as $ FF \approx CV^2 $  (Cox and Lewis, 1966; Nawrot
2010). For the Poisson Process, a special case of renewal processes, it holds that $ FF
= CV^2 = 1 $ . For processes with interval history (non-renewal) this equality is violated
(Nawrot 2010; Farkhooi et al., 2009, 2011; Avila-Akerberg and Chacron 2011). Any
strong deviation from the equality can be interpreted with respect to short and long
time scale variability as the ISI variability reflects variability on a short time scale (in
the order of the average ISI) while the $FF$ reflects variability on a long time scale (on
the order of the average separation of identical experimental trials).

The estimation of the CV is, however, hampered by the fact that single unit firing
rates change over time. Here, we make use of a local measure termed CV2 (Holtky
et al., 1996; Ponce-Alvarez et al., 2010). For any two consecutive inter spike intervals
ISI and ISI' encountered in a single trial we define the coefficient of variation for these
two intervals as
$$ m = 2\frac{\mid ISI-ISI^{\prime} \mid}{ISI + ISI^{\prime}}.$$

Collect these numbers for all consecutive intervals within the estimation window *W*
and across all trials. Then compute the average across all *m*-values:
$$ CV2 = \frac{1}{N} \sum_{i=1}^{N}m $$
This is your CV2 estimate for one trial ensemble.


Tasks
1. The results shall be presented in *Figure 3* with 1 x 3 panels (1 row of 3 panels). You may use `subplots` from `matplotlib.pyplot` to generate several axes within one figure.
2. To this end, loop across all data files for condition C1 aligned to TS. You may use the predefined list `SELECTED_C1_TS_FILES` or `glob`/`Path.glob` to search all files matching a pattern. For each single unit compute both the FF and $CV2^2$ for all six directions within one fixed estimation interval or window *W*. The longer this estimation window, the more robust the estimates of FF and CV2. Short estimation windows additionally introduce estimation biases (Nawrot et al., 2008). On the other hand, you want to restrict your analysis to a well-defined trial epoch. Inspect your results in Part I and define a window over a rather constant period of FF before the reaction signal RS, i.e. before the monkey moves the hand.
3. *Computation of $CV2$*.
    To compute $CV2$ for each trial ensemble, loop across directions and trials. For each trial detect all ISIs within *W* and compute the *m*-values for all consecutive intervals. Collect these *m*-values for all trials and then compute $CV2$. Repeat this for all six directions and keep, for each single unit, all six numbers.
4. *Computation of FF.*
    Again for each single unit, loop across all six directions. For any given trial ensemble the number of spikes in each trial is easily computed with `sum` on the binary spike matrix.
5. *Irregularity versus trial-by-trial variability*.
    Now scatter $CV2^2$ against FF in panel 1 of *Figure 3*. You might want to use logarithmic axes. Make sure that both axes appear with equal length in your figure (quadratic axes). Use the same limits for the x- and y-axes. You can also plot the renewal expectation as a red line.
6. *Marginal distributions*.
    Now compute the histograms of FF and $CV2^2$ across single units and plot them in panels 2 and 3 of *Figure 3*. Compare the two distributions and discuss whether the renewal prediction $FF \approx CV2^2$ seems to hold for this dataset.


You should be able to see clear deviations from the renewal expectation with $FF \gg CV2^2$. We may interpret this as a non-stationarity across trials that results from
different cortical network states (Nawrot et al., 2003; Nawrot 2010; Riehle et al. 2017)
and attractor network models have been suggested to account for an excessive Fano
factor (Deco et al., 2012; Litwin-Kumar and Doiron 2012; Mazzucato et al., 2015;
Rost et al., submitted). The reduction of the FF during task evolution (Part I) can
also be interpreted with these models and, in addition, might be a result of cellular
mechanisms (Farkhooi et al., 2013).

In [ ]:
def cv2_for_trials(spike_data):
    values = []
    for trial in range(spike_data.shape[1]):
        spike_times = np.flatnonzero(spike_data[:, trial])
        isi = np.diff(spike_times)
        if isi.size >= 2:
            values.extend(2 * np.abs(np.diff(isi)) / (isi[:-1] + isi[1:]))
    return np.mean(values) if values else np.nan


ff_values = []
cv2_squared_values = []
window_start_ms, window_end_ms = 0, 750

for path in SELECTED_C1_TS_FILES:
    sparse = load_sparse_format(path)
    start = round((window_start_ms - sparse.CutIntervalMS[0]) / sparse.TimeResolutionMS)
    end = round((window_end_ms - sparse.CutIntervalMS[0]) / sparse.TimeResolutionMS)
    for direction_data in sparse.Data:
        window_data = direction_data.toarray()[start:end]
        counts = window_data.sum(axis=0)
        ff_values.append(counts.var() / counts.mean() if counts.mean() > 0 else np.nan)
        cv2_squared_values.append(cv2_for_trials(window_data) ** 2)

ff_values = np.asarray(ff_values)
cv2_squared_values = np.asarray(cv2_squared_values)
valid = np.isfinite(ff_values) & np.isfinite(cv2_squared_values)
limit = 1.05 * max(ff_values[valid].max(), cv2_squared_values[valid].max())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(cv2_squared_values[valid], ff_values[valid])
axes[0].plot([0, limit], [0, limit], "r--", label="renewal expectation")
axes[0].set(xlim=(0, limit), ylim=(0, limit), aspect="equal",
            xlabel="$CV2^2$", ylabel="Fano factor")
axes[0].legend()
axes[1].hist(ff_values[valid], bins=12)
axes[1].set_xlabel("Fano factor")
axes[2].hist(cv2_squared_values[valid], bins=12)
axes[2].set_xlabel("$CV2^2$")
plt.tight_layout()
plt.show()

print("mean FF:", np.nanmean(ff_values))
print("mean CV2 squared:", np.nanmean(cv2_squared_values))


## **References**

Arieli A, Sterkin A, Grinvald A, Aertsen A. 1996. Dynamics of ongoing activity: explanation of
the large variability in evoked cortical responses. Science 273: 1868-1871


Avila-Akerberg O, Chacron MJ (2011) Nonrenewal spike train statistics: causes and
functional consequences on neural coding. Exp Brain Res (2011) 210:353–371


Churchland AK, Kiani R, Chaudhuri R, Wang X-J, Pouget A, Shadlen MN (2011) Variance as
a signature of neural computations during decision making. Neuron 69: 818-831.


Churchland MM, Yu BM, Ryu SI, Santhanam G, Shenoy KV (2006) Neural variability in
premotor cortex provides a signature of motor preparation. J Neurosci 26: 3697-3712.


Churchland MM, Yu BM, Cunningham JP, Sugrue LP, Cohen MR, Corrado GS, Newsome
WT, Clark AM, Hosseini P, Scott BB, Bradley DC, Smith MA, Kohn A, Movshon JA,
Armstrong KM, Moore T, Chang SW, Snyder LH, Lisberger SG, Priebe NJ, Finn IM, Ferster
D, Ryu SI, Santhanam G, Sahani M, Shenoy KV (2010) Stimulus onset quenches neural
variability: a widespread cortical phenomenon. Nat Neurosci 13: 369-378.


Cox DR, Lewis PAW. 1966. The statistical analysis of series of events. Monographs on
Applied probability and statistics. London: Chapman & Hall.


Deco G, Hugues E. 2012. Neural network mechanisms underlying stimulus driven variability
reduction. ​PLoS Comput Biol​ ​8​: e1002395.


Farkhooi F, Strube-Bloss MF, Nawrot MP (2009) Serial correlation in neural spike trains:
experimental evidence, stochastic modeling, and single neuron variability. Phys Rev E Stat
Nonlin Soft Matter Phys 79:021905


Farkhooi F, Muller E, Nawrot MP (2011) Adaptation reduces variability of the neuronal
population code. Phys Rev E 83: 050905.


Farkhooi F, Froese A, Muller E, Menzel R, Nawrot MP. 2013. Cellular adaptation facilitates
sparse and reliable coding in sensory pathways. ​PLoS Comput Biol​ 9: e1003251.


Holt GR, Softky WR, Koch C, Douglas RJ (1996) Comparison of discharge variability in vitro
and in vivo in cat visual cortex neurons. J Neurophysiol 75: 1806-1814.


Litwin-Kumar A, Doiron B. 2012. Slow dynamics and high variability in balanced cortical
networks with clustered connections. ​Nat Neurosci​ 15: 1498-1505.


Kass RE, Ventura V, Brown EN (2005) Statistical Issues in the Analysis of Neuronal Data. J
neurophysiol 94: 8-25


Mazzucato L, Fontanini A, La Camera G. 2015. Dynamics of multistable states during
ongoing and evoked cortical activity. ​J Neurosci​ 35: 8214-8231.


Nawrot MP. 2010. Analysis and interpretation of interval and count variability in neural spikes
trains. In: Grün S, Rotter S (eds) Analysis of parallel spikes trains. Springer Series in
Computational Neuroscience 7. Springer Verlag, New York, Berlin, pp 34-58.


Nawrot MP, Boucsein C, Rodriguez Molina V, Riehle A, Aertsen A, Rotter S. 2008.
Measurements of variability dynamics in cortical spike trains. J Neurosci Meth 169: 374-390.
Nawrot MP, Riehle A, Aertsen A, Rotter S (2003) Variability of motor cortical activity
explained by network dynamics on multiple time scales. In: Ongoing activity in cortical
networks: noise, variability and context (PhD thesis), URN: nbn:de:bsz:25-opus-73426.


Ponce-Alvarez A, Kilavik BE, Riehle A (2010) Comparison of local measures of spike time
irregularity and relating variability to firing rate in motor cortical neurons. J Comput Neurosci
29: 351-365


Rickert J, Riehle A, Aertsen A, Rotter S, Nawrot MP (2009) Dynamic encoding of movement
direction in motor cortical neurons. Journal of Neuroscience 29: 13870-13882


Riehle A, Vaadia E (Eds., 2005) Motor Cortex in Voluntary Movements: a distributed system
for distributed functions. CRC Press, Boca Raton, Florida, USA


Riehle, A., Brochier, T., Nawrot, M., & Grün, S.. (2018). Behavioral Context Determines
Network State and Variability Dynamics in Monkey Motor Cortex. Frontiers in Neural Circuits,
12, 52


Rost, T., Deger, M., & Nawrot, M. P.. (2018). Winnerless competition in clustered balanced
networks: inhibitory assemblies do the trick. Biological Cybernetics, 112(1-2), 81–98


Rostami A, Rost T, Riehle A, van Albada SJ, and Nawrot MP. Spiking neural network model
of motor cortex with joint excitatory and inhibitory clusters reflects task uncertainty, reaction
times, and variability dynamics. bioRxiv, 2020


Shadlen MN, Newsome WT. 1998. The variable discharge of cortical neurons: implications
for connectivity, computations, and information coding. J Neurosci 18: 3870-3896
